# 05 · Evaluation, calibration, uncertainty and explainability
Everything here is recomputed from the saved artifacts in `backend/models/`.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
BACKEND = (Path.cwd().parent / "backend").resolve()   # notebooks/ -> backend/
sys.path.insert(0, str(BACKEND))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from app.config import settings
pd.set_option("display.width", 140)

In [2]:
import joblib
from app.ml.evaluate import compute_metrics, bootstrap_ci, calibration_points
from app.ml.features import CSV_COLUMNS
model = joblib.load(settings.MODELS_DIR / settings.PIPELINE_FILE)
meta = json.loads((settings.MODELS_DIR / settings.METADATA_FILE).read_text())
test = pd.read_csv(settings.MODELS_DIR / "test_set.csv")
prob = model.predict_proba(test[CSV_COLUMNS])[:, 1]
thr = meta["threshold"]
print(meta["selected_model_display"], "| threshold", round(thr, 3), "|", meta["threshold_method"])

Support Vector Machine (RBF) | threshold 0.308 | Youden's J (sensitivity + specificity - 1) maximised on out-of-fold predictions from 5-fold CV on the training set; the test set was not used


## Test-set metrics (computed now, from the saved model)

In [3]:
m = compute_metrics(test.Outcome, prob, thr); m0 = compute_metrics(test.Outcome, prob, .5)
pd.DataFrame({"tuned threshold": m, "threshold 0.5": m0}).drop(["confusion_matrix", "n"])

,tuned threshold,threshold 0.5
threshold,0.3082,0.5
accuracy,0.7403,0.7078
precision,0.6,0.5957
recall,0.7778,0.5185
specificity,0.72,0.81
f1,0.6774,0.5545
roc_auc,0.8146,0.8146
pr_auc,0.682,0.682
brier,0.1745,0.1745


In [4]:
print("Confusion matrix (tuned threshold):", m["confusion_matrix"])
print("Bootstrap 95% CI:", bootstrap_ci(test.Outcome, prob))

Confusion matrix (tuned threshold): {'tn': 72, 'fp': 28, 'fn': 12, 'tp': 42}


Bootstrap 95% CI: {'n_bootstrap': 1000, 'roc_auc_95ci': [0.7446, 0.8754], 'brier_95ci': [0.1404, 0.2137]}


## ROC and precision–recall curves

In [5]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
RocCurveDisplay.from_predictions(test.Outcome, prob, ax=ax[0]); ax[0].plot([0, 1], [0, 1], "k--")
PrecisionRecallDisplay.from_predictions(test.Outcome, prob, ax=ax[1]); plt.show()

## Calibration
Brier score = mean squared error of the probabilities. The reliability diagram compares predicted probability with the observed rate per bin.

In [6]:
pts = pd.DataFrame(calibration_points(test.Outcome, prob))
plt.plot([0, 1], [0, 1], "k--"); plt.plot(pts.mean_predicted, pts.fraction_positive, marker="o")
plt.xlabel("mean predicted"); plt.ylabel("observed fraction diabetic"); plt.title(f"Brier = {m['brier']:.3f}"); plt.show()
ev = json.loads((settings.METRICS_DIR / "evaluation.json").read_text())
print(ev["calibration"]["decision"])

Calibration not applied: mean CV Brier 0.1525 with it vs 0.1520 without it, so the extra layer did not help.


## Uncertainty: bootstrap ensemble

In [7]:
from app.ml.uncertainty import bootstrap_uncertainty
boots = joblib.load(settings.MODELS_DIR / settings.BOOTSTRAP_FILE)
row = test[CSV_COLUMNS].iloc[[0]]
u = bootstrap_uncertainty(boots, row, thr, model.predict_proba(row)[0, 1])
print(u["explanation"]); print(ev["uncertainty"])

Across 50 bootstrap models, 90% of estimates fall between 48.7% and 70.2%. The whole range is on one side of the decision threshold, so the classification is stable under resampling of the training data.
{'n_models': 50, 'interval': '5th-95th percentile of bootstrap model estimates', 'test_mean_interval_width': 0.1622, 'test_median_interval_width': 0.1599, 'test_share_borderline': 0.2727, 'seconds': 48.0}


## SHAP: global and local explanations

In [8]:
g = json.loads((settings.METRICS_DIR / "shap_global.json").read_text())
pd.DataFrame(g["importance"])

,feature,label,mean_abs_shap,mean_shap
0,glucose,Glucose,0.17162,0.02718
1,bmi,BMI,0.08511,-0.00184
2,pregnancies,Pregnancies,0.04118,0.00181
3,diabetes_pedigree,Diabetes Pedigree Function,0.03298,0.00168
4,insulin,Insulin (2-hour serum),0.01719,0.00204
5,blood_pressure,Blood Pressure (diastolic),0.01634,0.00293
6,age,Age,0.01246,-0.00039
7,skin_thickness,Skin Thickness (triceps),0.00454,0.00069


In [9]:
from app.ml.explain import build_explainer, local_explanation
bg = joblib.load(settings.MODELS_DIR / settings.SHAP_BACKGROUND_FILE)
explainer = build_explainer(model, bg)
patient_raw = {k: v for k, v in zip(["pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","age"], row.iloc[0].tolist())}
e = local_explanation(explainer, row, model.predict_proba(row)[0, 1], patient_raw)
print("base", e["base_value"], "+ contributions =", e["sum_check"], "| model probability", e["prediction"])
pd.DataFrame(e["contributions"])[["label", "value", "shap_value", "direction"]]

base 0.32975 + contributions = 0.58113 | model probability 0.58113


,label,value,shap_value,direction
0,Glucose,159.000,0.29756,increases
1,BMI,27.400,-0.10231,decreases
2,Pregnancies,7.000,0.04903,increases
3,Insulin (2-hour serum),0.000,0.01991,increases
4,Diabetes Pedigree Function,0.294,-0.01400,decreases
5,Age,40.000,0.01304,increases
6,Skin Thickness (triceps),0.000,-0.00686,decreases
7,Blood Pressure (diastolic),64.000,-0.00499,decreases


> SHAP describes how the **model** used each feature. It is not evidence that a feature *causes* diabetes.

## What-if (model sensitivity)

In [10]:
base = test[CSV_COLUMNS].iloc[[0]].copy()
grid = pd.concat([base] * 30, ignore_index=True); grid["Glucose"] = np.linspace(56, 199, 30)
plt.plot(grid.Glucose, model.predict_proba(grid)[:, 1]); plt.axhline(thr, ls="--", c="r")
plt.xlabel("Glucose (others fixed)"); plt.ylabel("model probability"); plt.show()